<a href="https://colab.research.google.com/github/jorgemunozl/vla-test/blob/main/sixth_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 16 Test - Simulate Inference

Inference with the subtask code implementation, the text is now the most reasonable but it has some sense. The only issue is that the COMPLEMENTARY DATA is on blank. Updaat: You fix it by using validate features.

Deprecated since now we are dealing with the 05 lerobot version

In [1]:
import os

# Set this BEFORE importing pytorch/tensorflow
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import torch
print(torch.cuda.device_count()) 

1


In [2]:
DS_ID = "NONHUMAN-RESEARCH/general-task-index"
PRETRAINED_PATH = "google/paligemma2-3b-mix-224"

In [5]:
from xhuman.policies.pi05ki_pg2.configuration_pi05ki_pg2 import PI05KIPG2Config

policy_config = PI05KIPG2Config(repo_id="none",device="cuda",pretrained_path=PRETRAINED_PATH)

In [6]:
from xhuman.configs.default import LerobotDatasetConfig

dataset_config = LerobotDatasetConfig(
    repo_id=DS_ID,
)

In [7]:
from xhuman.configs.train import TrainPipelineConfigXHUMAN

cfg = TrainPipelineConfigXHUMAN(
    dataset=dataset_config,
    policy=policy_config,
)
cfg.validate()

In [8]:
from xhuman.datasets.factory import make_dataset_xhuman

cfg.dataset.train_with_subtasks = True
dataset = make_dataset_xhuman(cfg)


Fetching 29 files:   0%|          | 0/29 [00:00<?, ?it/s]

Fetching 29 files:   0%|          | 0/29 [00:00<?, ?it/s]

In [9]:
from xhuman.policies.factory import make_xhuman_policy

policy = make_xhuman_policy(
    cfg=cfg.policy,
    ds_meta=dataset.meta,
)

[pi05_ki_pg2] Stage 1: loading VLM weights from 'google/paligemma2-3b-mix-224'


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

[pi05_ki_pg2]   VLM weights loaded (missing=0, unexpected=0)
[pi05_ki_pg2] Stage 2: warm-starting from 'google/paligemma2-3b-mix-224'
[pi05_ki_pg2] WARNING: could not read pi05_base safetensors (OSError: google/paligemma2-3b-mix-224 does not appear to have a file named model.safetensors. Checkout 'https://huggingface.co/google/paligemma2-3b-mix-224/tree/main' for available files.); the model is returned with PaliGemma2 weights and a randomly initialized expert + flow heads.


In [11]:
from xhuman.policies.factory import make_xhuman_pre_post_processors

preprocessor, _ = make_xhuman_pre_post_processors(
        policy_cfg=policy_config,
    )

In [12]:
device = torch.device("cuda")

In [13]:
frame = dataset[0]
frame.keys()

dict_keys(['observation.images.left', 'observation.images.top', 'observation.images.right', 'action', 'observation.state', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index', 'general_task_index', 'action_is_pad', 'task', 'general_task', 'train_with_subtask'])

In [14]:
frame["observation.images.top"].shape

torch.Size([3, 376, 672])

In [ ]:
# Overwrite the language tokens
# frame["general_task"] = "Task: Move the fruits to the basket. Subtask: "

In [15]:
batch_subtask = preprocessor.subtask(frame)

In [16]:
batch_subtask.keys()

dict_keys(['action', 'next.reward', 'next.done', 'next.truncated', 'info', 'action_is_pad', 'task', 'index', 'task_index', 'episode_index', 'train_with_subtask', 'subtask', 'subtask_tokens', 'observation.images.left', 'observation.images.top', 'observation.images.right', 'observation.state', 'observation.language.tokens', 'observation.language.attention_mask', 'observation.subtask.tokens', 'observation.subtask.attention_mask'])

In [17]:
batch_subtask["task"]

['Task: pick the fruits from the table and place them in the basket. Subtask: ']

In [18]:
images, img_masks = policy._preprocess_images(batch_subtask)

In [19]:
tokens , masks = batch_subtask["observation.language.tokens"], batch_subtask["observation.language.attention_mask"]

In [20]:
max_decoding_steps = 200
eos_token_id = 1
temperature = 0.1

In [22]:
from xhuman.policies.utils import make_att_2d_masks
from torch.nn import functional as F

In [23]:
        prefix_embs, prefix_pad_masks, prefix_att_masks = policy.model.embed_prefix(
            images, img_masks, tokens, masks
        )

        prefix_att_2d_masks = make_att_2d_masks(
            prefix_pad_masks, prefix_att_masks
        )

In [27]:
prefix_position_ids = torch.cumsum(prefix_pad_masks, dim=1) - 1

In [28]:
        # Convert 2D attention masks to 4D format expected by the model
        prefix_att_2d_masks_4d = policy.model._prepare_attention_masks_4d(
            prefix_att_2d_masks
        )

        lang_model = policy.model.paligemma_with_expert.paligemma.model.language_model
        lang_model.config._attn_implementation = "eager"  # noqa: SLF001

        embeddings, past_key_values = policy.model.paligemma_with_expert.forward(
            attention_mask=prefix_att_2d_masks_4d,
            position_ids=prefix_position_ids,
            past_key_values=None,
            inputs_embeds=[prefix_embs, None],
            use_cache=True,
        )

        # embeddings[0] shape: (B, total_seq_len, embd_dim)
        # Extract last token: (B, embd_dim)
        last_token_embed = embeddings[0][:, -1, :]

        # Convert to logits: (B, vocab_size)
        last_logits = (
            policy.model.paligemma_with_expert.paligemma.lm_head(last_token_embed)
        )

        batch_size = last_logits.shape[0]
        device = last_logits.device

        # prefix_valid_length is the number of valid (non-padded) tokens
        prefix_valid_length = torch.sum(prefix_pad_masks, dim=1)  # (B,)
        output_tokens = torch.zeros((batch_size, max_decoding_steps),
                                    dtype=torch.long, device=device)
        all_eos = torch.zeros(batch_size, dtype=torch.bool, device=device)

        # Prefix attention mask plus the new token attention mask
        running_attention_mask = prefix_pad_masks.clone()

In [29]:
        # Autoregressive Loop
        for step in range(max_decoding_steps):
            # Sample next token
            if temperature > 0.0:
                probs = F.softmax(last_logits / temperature, dim=-1)
                # token shape: (B, 1)
                token = torch.multinomial(probs, num_samples=1)
            else:
                token = torch.argmax(last_logits, dim=-1, keepdim=True)

            output_tokens[:, step] = token.squeeze(-1)

            # Check for EOS
            all_eos |= (token.squeeze(-1) == eos_token_id)
            if all_eos.all():
                break

            # Feed the new token back in the model
            # token shape: (B, 1) -> embed_tokens returns (B, 1, embd_dim)
            next_token_embeds = lang_model.embed_tokens(token)

            # Create position_ids for the new token
            position_ids = prefix_valid_length[:, None] + step

            # Create attention mask for the new token
            new_mask = torch.ones(
                (batch_size, 1),
                dtype=running_attention_mask.dtype,
                device=device
            )

            running_attention_mask = torch.cat(
                [running_attention_mask, new_mask], dim=1)

            # The attention mask can be 2d or 4d.
            embeds_list, past_key_values = policy.model.paligemma_with_expert.forward(
                inputs_embeds=[next_token_embeds, None],
                attention_mask=running_attention_mask,
                position_ids=position_ids,
                past_key_values=past_key_values,
                use_cache=True,
            )

            print(running_attention_mask)
            prefix_output = embeds_list[0]
            last_token_embed = prefix_output[:, -1, :]
            last_logits = (policy.model.paligemma_with_expert.paligemma.lm_head(last_token_embed))

tensor([[ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  

In [30]:
output_tokens

tensor([[234323,      1,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,    

In [31]:
policy._detokenize_subtask(output_tokens)

'myſelf'

In [32]:
policy._detokenize_subtask(tokens)

'Task: pick the fruits from the table and place them in the basket. Subtask:'